In [ ]:
# =====================================================================
# [Step 0] 환경 설정 및 필수 패키지
# =====================================================================
!pip install -q transformers datasets scikit-learn accelerate google-genai -U

import os
import re
import glob
import random
import time
import json
import getpass
import shutil
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from google.colab import files
from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.cluster import KMeans
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoModel, Trainer, TrainingArguments
from google import genai

# 재현성을 위한 시드 고정
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
torch.cuda.manual_seed_all(RANDOM_SEED)

# GitHub 저장소 클론
if not os.path.exists('/content/hansi-database'):
    !git clone https://github.com/chjk86/hansi-database.git /content/hansi-database

# =====================================================================
# [Step 1] 경로 설정 및 고정 평가 데이터(Test Set) 로드
# =====================================================================
FRONTIER_FILE = "/content/hansi-database/2026_DH_poster/변새시_2차정리본.txt"
MUNZIP_DIR = "/content/hansi-database/2025_munzip_title_text_ver"

CHECKPOINT_DIR = "/content/bert_checkpoints"
OUTPUT_DIR = "/content/bert_frontier_classifier_with_markup"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# [핵심] 인간 평가가 완료된 고정 테스트셋 로드
HUMAN_EVAL_FILE = "/content/00_final_integrated_evaluation_인간평가통합본.csv"
if not os.path.exists(HUMAN_EVAL_FILE):
    raise FileNotFoundError(f"[오류] {HUMAN_EVAL_FILE} 파일이 없습니다. Colab에 업로드하십시오.")

human_eval_df = pd.read_csv(HUMAN_EVAL_FILE)
test_texts_set = set(human_eval_df['시_원문'].tolist())

print(f"\n[진행] 인간 평가가 완료된 고정 테스트셋 {len(test_texts_set)}건을 로드했습니다.")

# =====================================================================
# [Step 2] 텍스트 파싱 및 훈련 데이터셋(Train/Val) 구축
# =====================================================================
def extract_pure_hanzi(text):
    clean_text = re.sub(r'<[^>]+>', '', text)
    return re.sub(r'[^\u4E00-\u9FFF]', '', clean_text)

def get_tag_content(block, tag_name):
    start_tag = f"<{tag_name}>"
    end_tag = f"</{tag_name}>"
    block_lower = block.lower()
    if start_tag.lower() in block_lower and end_tag.lower() in block_lower:
        start_idx = block_lower.find(start_tag.lower()) + len(start_tag.lower())
        end_idx = block_lower.find(end_tag.lower(), start_idx)
        return block[start_idx:end_idx].strip()
    return "없음"

frontier_train_pool = []
frontier_test_meta = {}

print("[진행] 훈련용 데이터를 파싱합니다 (테스트셋 원문은 훈련에서 배제됨)...")

# 정답 데이터(Label 1) 추출
try:
    with open(FRONTIER_FILE, 'r', encoding='utf-8') as f:
        content = f.read()
    
    for block in content.split("</Poem>"):
        if "<Poem" not in block and "<poem" not in block.lower(): continue
        
        text_val = get_tag_content(block, "text")
        hanzi = extract_pure_hanzi(text_val)
        
        if hanzi and len(hanzi) >= 10:
            term_val = get_tag_content(block, "term")
            allusion_val = get_tag_content(block, "allusion")
            evidence_val = get_tag_content(block, "evidence")
            meta = f"핵심시어: {term_val} | 전고: {allusion_val} | 판별근거: {evidence_val}"
            
            # 테스트셋에 포함된 텍스트면 메타데이터만 별도 보관 (훈련에서 제외)
            if hanzi in test_texts_set:
                frontier_test_meta[hanzi] = meta
            else:
                frontier_train_pool.append({'text': hanzi, 'metadata': meta, 'labels': 1})
except Exception as e:
    raise FileNotFoundError(f"[오류] 파일 파싱 실패. 경로를 확인하십시오. 상세: {e}")

TARGET_TRAIN_COUNT = len(frontier_train_pool)
non_frontier_train_pool = []

# 대조군 데이터(Label 0) 수집
munzip_files = glob.glob(os.path.join(MUNZIP_DIR, "*.txt")) + glob.glob(os.path.join(MUNZIP_DIR, "*.xml"))
random.shuffle(munzip_files)

for file_path in munzip_files:
    if len(non_frontier_train_pool) >= TARGET_TRAIN_COUNT: break
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            content = f.read()
            
        parts = content.lower().split("<text>")
        for i in range(1, len(parts)):
            if "</text>" in parts[i]:
                start_idx = content.lower().find("<text>", content.lower().find(parts[i][:10])) + 6
                end_idx = content.lower().find("</text>", start_idx)
                hanzi = extract_pure_hanzi(content[start_idx:end_idx])
                
                # 테스트셋에 포함되지 않은 새로운 시만 훈련 데이터로 편입
                if len(hanzi) >= 10 and (hanzi not in test_texts_set):
                    non_frontier_train_pool.append({
                        'text': hanzi, 
                        'metadata': "핵심시어: 없음 | 전고: 없음 | 판별근거: 없음", 
                        'labels': 0
                    })
    except Exception: pass

train_pool_df = pd.DataFrame(frontier_train_pool + non_frontier_train_pool)
print(f"[훈련 풀 구성 완료] 훈련용 변새시(1): {len(frontier_train_pool)}수 | 훈련용 비변새시(0): {len(non_frontier_train_pool)}수")

# =====================================================================
# [Step 3] 데이터 분할 (Train/Val/Test 확립)
# =====================================================================
# 전체 Train Pool에서 검증셋(Val)을 15% 분할
train_df, val_df = train_test_split(train_pool_df, test_size=0.15, random_state=RANDOM_SEED, stratify=train_pool_df['labels'])

# Test DataFrame 구축 (인간 평가 파일 기준 + 메타데이터 매핑)
test_data_list = []
for _, row in human_eval_df.iterrows():
    t_text = row['시_원문']
    t_label = row['실제_정답(0:비변새시, 1:변새시)']
    t_meta = frontier_test_meta.get(t_text, "핵심시어: 없음 | 전고: 없음 | 판별근거: 없음")
    test_data_list.append({'text': t_text, 'metadata': t_meta, 'labels': t_label})

test_df = pd.DataFrame(test_data_list)

train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)
test_dataset = Dataset.from_pandas(test_df)

# =====================================================================
# [Step 4] BERT 모델 파인튜닝 (지도 학습)
# =====================================================================
print("\n========== [1/4] BERT 파인튜닝 시작 ==========")
MODEL_NAME = "ethanyt/guwenbert-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_function(examples):
    tokenized = tokenizer(
        examples["text"], 
        examples["metadata"], 
        padding="max_length", 
        truncation=True, 
        max_length=256
    )
    if "token_type_ids" in tokenized: 
        del tokenized["token_type_ids"]
    return tokenized

tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_val = val_dataset.map(tokenize_function, batched=True)
tokenized_test = test_dataset.map(tokenize_function, batched=True)

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, predictions),
        "precision": precision_score(labels, predictions, zero_division=0),
        "recall": recall_score(labels, predictions, zero_division=0),
        "f1": f1_score(labels, predictions, average='binary', zero_division=0)
    }

training_args = TrainingArguments(
    output_dir=CHECKPOINT_DIR,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    learning_rate=3e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    label_smoothing_factor=0.1,
    remove_unused_columns=True,
    seed=RANDOM_SEED
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    compute_metrics=compute_metrics,
)

trainer.train()

model_save_path = os.path.join(OUTPUT_DIR, "state_dict_markup_bert.pt")
torch.save(model.state_dict(), model_save_path)
tokenizer.save_pretrained(OUTPUT_DIR)
zip_path = shutil.make_archive(OUTPUT_DIR, 'zip', OUTPUT_DIR)
files.download(zip_path)

print("\n[테스트셋 추론] 파인튜닝 모델 예측값을 추출합니다.")
predictions_output = trainer.predict(tokenized_test)
test_df['BERT_파인튜닝_예측'] = np.argmax(predictions_output.predictions, axis=-1)

# =====================================================================
# [Step 5] BERT 임베딩 군집화 (비지도 학습)
# =====================================================================
print("\n========== [2/4] BERT 비지도 군집화 시작 ==========")
texts_for_eval = test_df['text'].tolist()
true_labels = test_df['labels'].tolist()

unsup_model = AutoModel.from_pretrained(MODEL_NAME)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
unsup_model.to(device)
unsup_model.eval()

embeddings = []
with torch.no_grad():
    for i in range(0, len(texts_for_eval), 16):
        batch_texts = texts_for_eval[i:i+16]
        inputs = tokenizer(batch_texts, padding="max_length", truncation=True, max_length=150, return_tensors="pt").to(device)
        cls_embeddings = unsup_model(**inputs).last_hidden_state[:, 0, :].cpu().numpy()
        embeddings.extend(cls_embeddings)

kmeans = KMeans(n_clusters=2, random_state=RANDOM_SEED, n_init=10)
cluster_labels = kmeans.fit_predict(np.array(embeddings))

match_1_acc = accuracy_score(true_labels, cluster_labels)
match_2_acc = accuracy_score(true_labels, 1 - cluster_labels)
test_df['BERT_비지도군집_예측'] = cluster_labels if match_1_acc >= match_2_acc else 1 - cluster_labels

# =====================================================================
# [Step 6] LLM 제로샷 & 퓨샷 평가 (Gemini)
# =====================================================================
print("\n========== [3/4 & 4/4] LLM (Gemini) 평가 시작 ==========")
GEMINI_API_KEY = getpass.getpass("Gemini API 키를 붙여넣고 Enter를 누르십시오: ")
client = genai.Client(api_key=GEMINI_API_KEY)
LLM_MODEL_ID = 'gemini-3.5-flash'

def parse_json_response(response_text):
    cleaned = re.sub(r'^```(?:json)?\s*', '', response_text.strip(), flags=re.IGNORECASE)
    cleaned = re.sub(r'\s*```$', '', cleaned)
    try:
        return json.loads(cleaned)
    except:
        return {"prediction": 0, "reason": "JSON 파싱 실패"}

def evaluate_llm(texts, prompt_func):
    predictions, reasons = [], []
    for i, text in enumerate(texts):
        prompt = prompt_func(text)
        try:
            response = client.models.generate_content(model=LLM_MODEL_ID, contents=prompt)
            result = parse_json_response(response.text)
            predictions.append(int(result.get("prediction", 0)))
            reasons.append(result.get("reason", "근거 누락"))
            time.sleep(1.5)
        except Exception as e:
            predictions.append(0)
            reasons.append(f"API 오류: {e}")
        if (i + 1) % 10 == 0:
            print(f" - 진행 상황: {i + 1}/{len(texts)} 완료")
    return predictions, reasons

def get_zeroshot_prompt(text):
    return f"""당신은 한국 고전문학 및 한문학에 정통한 연구자입니다.
다음 한시(漢詩) 원문을 읽고, 이 작품이 변새시(邊塞詩)인지 판별하십시오.
변새시는 변방의 군사적 충돌, 이국적 풍경, 종군(從軍)의 고충 등을 다루는 장르입니다.
원문: {text}
반드시 아래 JSON 형식으로만 응답하십시오. (마크다운 금지)
{{"prediction": 1, "reason": "판별 근거"}}"""

def get_fewshot_prompt(text):
    return f"""당신은 한국 고전문학 및 한문학에 정통한 연구자입니다.
다음 한시(漢詩) 원문을 읽고, 이 작품이 변새시(邊塞詩)인지 판별하십시오.

[예시 1: 변새시 (Label 1)]
원문: 談笑腥塵一掃空海中蛟鰐識威風指揮能事天應惜合散奇謀鬼不功夜報捷書靈夏外春收烽火浿江東尙餘蟣蝨逃湯鼎願借丹衡扇祝融
응답: {{"prediction": 1, "reason": "捷書, 烽火 등 시어 사용, 장수의 위풍과 군사적 기모 묘사."}}

[예시 2: 비변새시 (Label 0)]
원문: 佳麗金陵聖祖居丹靑想像淚沾裾石頭獅子雲霞古玄武瓜洲舟楫踈一代衣冠宮殿在百年日月山河虛中天漢祚昔人語何迺于今久寂如
응답: {{"prediction": 0, "reason": "회고(懷古)시로 변방 군사적 대립과 무관함."}}

원문: {text}
반드시 아래 JSON 형식으로만 응답하십시오. (마크다운 금지)
{{"prediction": 1, "reason": "판별 근거"}}"""

print("\n[제로샷 평가 진행 중...]")
test_df['LLM_제로샷_예측'], test_df['LLM_제로샷_근거'] = evaluate_llm(texts_for_eval, get_zeroshot_prompt)
print("\n[퓨샷 평가 진행 중...]")
test_df['LLM_퓨샷_예측'], test_df['LLM_퓨샷_근거'] = evaluate_llm(texts_for_eval, get_fewshot_prompt)

# =====================================================================
# [Step 7] 원본 CSV에 예측 결과 병합 및 시각화
# =====================================================================
for col in ['BERT_파인튜닝_예측', 'BERT_비지도군집_예측', 'LLM_제로샷_예측', 'LLM_제로샷_근거', 'LLM_퓨샷_예측', 'LLM_퓨샷_근거']:
    human_eval_df[col] = test_df[col]

csv_output = '/content/final_integrated_evaluation_with_models.csv'
human_eval_df.to_csv(csv_output, index=False, encoding='utf-8-sig')

fig, axes = plt.subplots(2, 2, figsize=(12, 11))
fig.suptitle('Frontier Poetry Classification: LLM vs BERT (Fixed Human Eval Set)', fontsize=18, y=0.95)
CLASS_NAMES = ['Non-Frontier (0)', 'Frontier (1)']
TRUE_COL = '실제_정답(0:비변새시, 1:변새시)'

plot_data = [
    (axes[0, 0], confusion_matrix(test_df['labels'], test_df['BERT_파인튜닝_예측']), 'BERT: Fine-tuned (Supervised)'),
    (axes[0, 1], confusion_matrix(test_df['labels'], test_df['BERT_비지도군집_예측']), 'BERT: Clustering (Unsupervised)'),
    (axes[1, 0], confusion_matrix(test_df['labels'], test_df['LLM_제로샷_예측']), 'LLM: Zero-shot (Gemini 3.5)'),
    (axes[1, 1], confusion_matrix(test_df['labels'], test_df['LLM_퓨샷_예측']), 'LLM: Few-shot (Gemini 3.5)')
]

for ax, cm, title in plot_data:
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
                annot_kws={"size": 16}, ax=ax)
    
    tn, fp, fn, tp = cm.ravel()
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    
    ax.set_title(f"{title}\n(F1: {f1:.3f} | Prec: {precision:.3f} | Rec: {recall:.3f})", fontsize=13, pad=12)
    ax.set_ylabel('True Label', fontsize=12)
    ax.set_xlabel('Predicted Label', fontsize=12)

plt.tight_layout(rect=[0, 0, 1, 0.93])

png_output = '/content/confusion_matrix_2x2_human_eval_fixed.png'
plt.savefig(png_output, dpi=300, bbox_inches='tight', transparent=False)
plt.close()

print(f"\n[최종 완성] 원본 파일에 예측 결과를 병합한 시트 및 오차 행렬 이미지를 다운로드합니다.")
files.download(csv_output)
files.download(png_output)
